# Fashion compatibility: data exploration and analysis

## Author
- **Author:** Francesco Tedesco
- **Email:** francescotedesco7d2@gmail.com
- [**LinkedIn**](https://www.linkedin.com/in/francescotedesco7d2/)

## Overview
This notebook explores the outfits and products in order to decide which outfits are valid training examples: outfit sizes, product descriptions, the outfit rules, the selected outfits, and the overlap of products between outfits that rules out a held-out split.

In [1]:
import os
import sys
from pathlib import Path

# Make `src` importable when the kernel starts in `notebooks/`.
ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

from src.data_processing.group_manipulation import (
    create_combinations,
    create_configurations,
    get_grouped_counts,
    get_grouped_counts_feature_values,
    get_unique_sets_features,
    select_valid_outfits,
)
from src.data_processing.product_processing import get_des_product_class
from src.data_processing.schema import PRODUCT_CLASSES
from src.generation.candidates import catalogues, image_year
from src.utils.setup_utilities import load_config

# Optional path to an alternative config file; defaults to `config.yml`.
config = load_config(os.environ.get("FASHION_CONFIG"))

In [2]:
df_outfits = pd.read_csv(config["data"]["outfits_path"], dtype={"cod_modelo_color": str})
df_products = pd.read_csv(config["data"]["products_path"], dtype={"cod_modelo_color": str})
df_outfit_products = pd.merge(df_outfits, df_products, on="cod_modelo_color", how="outer")

## Initial exploration

In [3]:
df_outfits.head()

,cod_outfit,cod_modelo_color
0,1,51000622-02
1,1,43067759-01
2,1,53060518-02
3,1,53030594-08
4,1,43077762-01


`df_outfits` lists the products (`cod_modelo_color`) of each outfit (`cod_outfit`).

In [4]:
df_products.head()

,cod_modelo_color,cod_color_code,des_color_specification_esp,des_agrup_color_eng,des_sex,des_age,des_line,des_fabric,des_product_category,des_product_aggregated_family,des_product_family,des_product_type,des_filename
0,41085800-02,02,OFFWHITE,WHITE,Female,Adult,SHE,P-PLANA,Bottoms,Trousers & leggings,Trousers,Trousers,datathon/images/2019_41085800_02.jpg
1,53000586-TO,TO,TEJANO OSCURO,BLUE,Female,Adult,SHE,J-JEANS,Bottoms,Jeans,Jeans,Jeans,datathon/images/2019_53000586_TO.jpg
2,53030601-81,81,ROSA PASTEL,PINK,Female,Adult,SHE,P-PLANA,"Dresses, jumpsuits and Complete set",Dresses and jumpsuits,Dresses,Dress,datathon/images/2019_53030601_81.jpg
3,53050730-15,15,MOSTAZA,YELLOW,Female,Adult,SHE,P-PLANA,"Dresses, jumpsuits and Complete set",Dresses and jumpsuits,Dresses,Dress,datathon/images/2019_53050730_15.jpg
4,53070773-70,70,ROJO,RED,Female,Adult,SHE,P-PLANA,Tops,Shirts,Shirt,Shirt,datathon/images/2019_53070773_70.jpg


`df_products` describes each product, including the path of its image.

In [5]:
products = set(df_products["cod_modelo_color"])
products_in_outfits = set(df_outfits["cod_modelo_color"])
print("Outfits:", df_outfits["cod_outfit"].nunique())
print("Products:", len(products))
print("Products without an outfit:", len(products - products_in_outfits))

Outfits: 7842
Products: 9222
Products without an outfit: 0


Number of outfits by size (number of products):

In [6]:
get_grouped_counts(df_outfits, "cod_outfit", "outfit")

,outfit_size,grouped_outfit_size
0,2,30
1,3,142
2,4,758
3,5,3875
4,6,1737
5,7,674
6,8,309
7,9,174
8,10,95
9,11,26


Outfits with two products, and those with ten or more, are rare: they are candidates for exclusion.

In [7]:
df_products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9222 entries, 0 to 9221
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   cod_modelo_color               9222 non-null   object
 1   cod_color_code                 9222 non-null   object
 2   des_color_specification_esp    9222 non-null   object
 3   des_agrup_color_eng            9222 non-null   object
 4   des_sex                        9222 non-null   object
 5   des_age                        9222 non-null   object
 6   des_line                       9222 non-null   object
 7   des_fabric                     9222 non-null   object
 8   des_product_category           9222 non-null   object
 9   des_product_aggregated_family  9222 non-null   object
 10  des_product_family             9222 non-null   object
 11  des_product_type               9222 non-null   object
 12  des_filename                   9222 non-null   object
dtypes: 

## Product descriptions

Not all outfits are suitable for training, so the product descriptions are examined first to decide which of them help define valid outfits.

In [8]:
df_products["des_product_category"].unique()

array(['Bottoms', 'Dresses, jumpsuits and Complete set', 'Tops',
       'Accesories, Swim and Intimate', 'Outerwear', 'Beauty', 'Home'],
      dtype=object)

In [9]:
df_products["des_product_family"].unique()

array(['Trousers', 'Jeans', 'Dresses', 'Shirt', 'Sweater', 'Skirts',
       'Jewellery', 'Bags', 'Glasses', 'Wallets & cases', 'Shorts',
       'Tops', 'Belts and Ties', 'Jumpsuit', 'Jackets', 'Coats',
       'Footwear', 'Hats, scarves and gloves', 'T-shirt', 'Blazers',
       'Gadgets', 'Swimwear', 'Vest', 'Fragances', 'Cardigans',
       'Trenchcoats', 'Puffer coats', 'Outer Vest',
       'Leggings and joggers', 'Deco Accessories', 'Poloshirts',
       'Intimate', 'Sweatshirts', 'Deco Textiles', 'Bedding', 'Bodysuits',
       'Leather jackets', 'Parkas', 'Glassware'], dtype=object)

In [10]:
df_products["des_product_type"].unique()

array(['Trousers', 'Jeans', 'Dress', 'Shirt', 'Sweater', 'Skirt',
       'Earrings', 'Totes bag', 'Sunglasses', 'Card holder', 'Wallet',
       'Shorts', 'Top', 'Belt', 'Crossbody bag', 'Jumpsuit', 'Jacket',
       'Coat', 'Sandals', 'Kerchief', 'Shoes', 'Blouse', 'T-Shirt',
       'Blazer', 'Umbrella', 'Citybag', 'Bikini top', 'Vest',
       'Shoulder bag', 'Bodymist', 'Beanie', 'Handbag', 'Cardigan',
       'Glasses', 'Trenchcoat', 'Puffer coat', 'Necklace',
       'Bikini pantie', 'Outer vest', 'Scarf', 'Ankle Boots', 'Leggings',
       'Basket', 'Cosmetic bag', 'Ring', 'Poloshirt', 'Pyjama',
       'Sweatshirt', 'Plaid', 'Boots', 'Hat', 'Duvet Covers',
       'Beach Towel', 'Gloves', 'Bodysuit', 'Fragance', 'Leather Jacket',
       'Hairband', 'Bermudas', 'Cap', 'Parka', 'Pyjama Trousers',
       'Pyjama Shirt', 'Bras', 'Trainers', 'Foulard', 'Hairclip', 'Case',
       'Bracelet', 'Pyjama Shorts', 'Sweater Vest', 'Pyjiama Sweater',
       'Bucket bag', 'Jacket (Cazadora)', 'Purse',

The more specific the description, the more labels it has. Counts of the most general one, `des_product_category`:

In [11]:
df_outfit_products["des_product_category"].value_counts()

des_product_category
Accesories, Swim and Intimate          27315
Tops                                    6313
Bottoms                                 5839
Outerwear                               2080
Dresses, jumpsuits and Complete set     1777
Home                                     169
Beauty                                    89
Name: count, dtype: int64

Home and Beauty are rare and are excluded from the outfit rules. The most common category, *Accesories, Swim and Intimate*, groups very different products, so its families are examined next.

In [12]:
df_accesories = df_outfit_products[df_outfit_products["des_product_category"] == "Accesories, Swim and Intimate"]
df_accesories["des_product_family"].value_counts()

des_product_family
Jewellery                   9686
Footwear                    7976
Bags                        6152
Glasses                      910
Belts and Ties               767
Wallets & cases              611
Hats, scarves and gloves     551
Intimate                     473
Swimwear                     158
Gadgets                       31
Name: count, dtype: int64

The jewellery family also groups different products; its types are examined next.

In [13]:
df_jewellery = df_accesories[df_accesories["des_product_family"] == "Jewellery"]
df_jewellery["des_product_type"].value_counts()

des_product_type
Earrings    5858
Ring        1832
Necklace    1363
Bracelet     614
Hairclip      19
Name: count, dtype: int64

The three descriptions are combined into one product class: the family for accessories other than jewellery, the type for jewellery, and the category otherwise.

In [14]:
df_outfit_products = df_outfit_products.copy()
df_outfit_products["des_product_class"] = df_outfit_products.apply(get_des_product_class, axis=1)
df_outfit_products["des_product_class"].unique()

array(['Footwear', 'Tops', 'Bags', 'Earrings', 'Bottoms', 'Outerwear',
       'Dresses, jumpsuits and Complete set', 'Necklace',
       'Hats, scarves and gloves', 'Belts and Ties', 'Bracelet',
       'Swimwear', 'Glasses', 'Ring', 'Gadgets', 'Wallets & cases',
       'Home', 'Intimate', 'Beauty', 'Hairclip'], dtype=object)

## Outfit structure

`top_sets` returns the most frequent multisets of a description among the outfits of one size. First, the product classes for every size:

In [15]:
grouped = {}


def top_sets(size, feature, top=5):
    """Most frequent multisets of `feature` values among outfits with `size` products."""
    if feature not in grouped:
        grouped[feature] = get_grouped_counts_feature_values(
            df_outfit_products, "cod_outfit", "outfit", feature
        )
    table = grouped[feature]
    outfits = table.loc[table["outfit_size"] == size, f"cod_outfit_{feature}_tuple"].iloc[0]
    return get_unique_sets_features(outfits, feature).head(top)


for size in range(2, 11):
    print(f"Outfits with {size} products")
    display(top_sets(size, "des_product_class"))

Outfits with 2 products


,des_product_class_set,des_product_class_outfit_codes,des_product_class_count
1,{'Wallets & cases': 2},"[1571, 1572, 1573, 1574, 1580, 1620, 1621, 203...",14
3,{'Intimate': 2},"[2711, 2727, 5794, 5795]",4
4,"{'Tops': 1, 'Bottoms': 1}","[3712, 3755, 3818, 7373]",4
0,"{'Tops': 1, 'Earrings': 1}","[1214, 3872]",2
5,"{'Footwear': 1, 'Intimate': 1}","[4034, 4496]",2


Outfits with 3 products


,des_product_class_set,des_product_class_outfit_codes,des_product_class_count
11,"{'Dresses, jumpsuits and Complete set': 1, 'Ea...","[942, 1333, 1504, 1582, 1594, 1809, 2068, 2094...",18
4,{'Wallets & cases': 3},"[591, 810, 813, 990, 1384, 1557, 1684, 1759, 1...",13
24,"{'Hats, scarves and gloves': 3}","[1318, 1319, 1320, 1481, 1482, 1483, 2091, 209...",9
15,"{'Dresses, jumpsuits and Complete set': 1, 'Ba...","[1019, 1593, 3368, 5464, 5479, 6421, 6949, 696...",9
17,"{'Glasses': 1, 'Bags': 1, 'Footwear': 1}","[1141, 1536, 1567, 1709, 1710, 2102, 2335, 6417]",8


Outfits with 4 products


,des_product_class_set,des_product_class_outfit_codes,des_product_class_count
3,"{'Dresses, jumpsuits and Complete set': 1, 'Ea...","[55, 160, 191, 229, 231, 232, 249, 251, 252, 2...",210
2,"{'Bottoms': 1, 'Tops': 1, 'Earrings': 1, 'Foot...","[39, 185, 288, 350, 352, 369, 425, 428, 463, 4...",90
16,"{'Glasses': 1, 'Bags': 1, 'Footwear': 1, 'Ring...","[609, 709, 895, 982, 987, 988, 1037, 1048, 107...",42
9,"{'Tops': 1, 'Bottoms': 1, 'Bags': 1, 'Footwear...","[312, 366, 570, 574, 688, 701, 789, 804, 917, ...",34
26,"{'Earrings': 1, 'Belts and Ties': 1, 'Footwear...","[883, 888, 964, 1008, 1009, 1092, 1385, 1597, ...",21


... (8 outputs left out)

Outfits with 9 products


,des_product_class_set,des_product_class_outfit_codes,des_product_class_count
36,"{'Earrings': 4, 'Ring': 3, 'Necklace': 2}","[1045, 1046, 1047, 1264, 1417, 1849, 2050, 226...",9
72,"{'Bags': 1, 'Footwear': 2, 'Tops': 3, 'Bottoms...","[3038, 3102, 3211]",3
86,"{'Footwear': 2, 'Tops': 3, 'Bags': 1, 'Earring...","[4101, 4102, 6270]",3
85,"{'Bags': 2, 'Tops': 2, 'Outerwear': 1, 'Footwe...","[4074, 5230, 5231]",3
1,"{'Bottoms': 1, 'Outerwear': 1, 'Tops': 1, 'Ear...","[47, 421, 765]",3


Outfits with 10 products


,des_product_class_set,des_product_class_outfit_codes,des_product_class_count
44,"{'Bags': 2, 'Wallets & cases': 8}","[2606, 2637, 4002, 4454, 4456, 4960, 5661, 5662]",8
2,"{'Bottoms': 1, 'Earrings': 1, 'Necklace': 2, '...","[92, 204, 607, 628]",4
6,"{'Earrings': 1, 'Bags': 2, 'Bracelet': 1, 'Top...","[297, 348]",2
58,"{'Bags': 1, 'Tops': 3, 'Bottoms': 1, 'Belts an...","[4083, 4084]",2
19,"{'Tops': 2, 'Bottoms': 1, 'Earrings': 2, 'Neck...","[1024, 1235]",2


For outfits with five products, the most frequent sets of each of the three dataset descriptions:

In [16]:
for feature in ("des_product_category", "des_product_family", "des_product_type"):
    display(top_sets(5, feature))

,des_product_category_set,des_product_category_outfit_codes,des_product_category_count
0,"{'Bottoms': 1, 'Accesories, Swim and Intimate'...","[2, 9, 10, 26, 31, 41, 42, 52, 63, 66, 75, 100...",1872
1,"{'Dresses, jumpsuits and Complete set': 1, 'Ac...","[15, 22, 23, 24, 32, 33, 34, 37, 38, 56, 57, 7...",819
3,"{'Tops': 1, 'Bottoms': 1, 'Outerwear': 1, 'Acc...","[46, 78, 119, 120, 166, 168, 169, 194, 196, 21...",494
2,"{'Bottoms': 1, 'Tops': 2, 'Accesories, Swim an...","[29, 287, 673, 771, 803, 857, 1104, 1111, 1338...",195
4,"{'Bottoms': 1, 'Outerwear': 1, 'Accesories, Sw...","[113, 179, 374, 452, 477, 599, 604, 891, 944, ...",123


,des_product_family_set,des_product_family_outfit_codes,des_product_family_count
2,"{'Dresses': 1, 'Bags': 1, 'Footwear': 1, 'Jewe...","[15, 22, 23, 24, 32, 33, 34, 37, 38, 56, 57, 7...",417
3,"{'Shirt': 1, 'Trousers': 1, 'Jewellery': 1, 'F...","[26, 31, 138, 227, 311, 387, 443, 465, 865, 11...",164
235,"{'Bags': 1, 'Jewellery': 1, 'Dresses': 1, 'Foo...","[2592, 2627, 2628, 2631, 2644, 2684, 2694, 269...",161
55,"{'T-shirt': 1, 'Jewellery': 1, 'Jeans': 1, 'Ba...","[483, 620, 624, 629, 633, 733, 738, 778, 1130,...",135
8,"{'Trousers': 1, 'Sweater': 1, 'Bags': 1, 'Jewe...","[52, 66, 494, 504, 691, 693, 696, 699, 700, 70...",135


,des_product_type_set,des_product_type_outfit_codes,des_product_type_count
3,"{'Dress': 1, 'Handbag': 1, 'Sandals': 1, 'Earr...","[15, 22, 23, 24, 33, 37, 153, 163, 228, 353, 3...",54
444,"{'Crossbody bag': 1, 'Dress': 1, 'Ring': 1, 'S...","[2600, 2601, 2804, 2914, 3336, 3538, 4052, 426...",35
7,"{'Necklace': 1, 'Dress': 1, 'Handbag': 1, 'Ear...","[32, 57, 324, 516, 645, 825, 862, 1016, 1231, ...",35
440,"{'Shoulder bag': 1, 'Earrings': 1, 'Dress': 1,...","[2592, 2631, 2821, 3405, 3448, 3480, 3685, 401...",32
459,"{'Sunglasses': 1, 'Sandals': 1, 'Earrings': 1,...","[2627, 2644, 2699, 2776, 2778, 2817, 2823, 346...",31


## Valid outfits

These observations define the rules for selecting outfits:

1. The base of an outfit is one of these sets:
   - Tops + Bottoms + Footwear + Earrings
   - Dress + Footwear + Earrings
2. The outfit can add any of these complements, each at most once: Outerwear, Bags, Glasses, Ring and Necklace.

A valid outfit is a base plus any subset of the complements: 2 × 2⁵ = 64 configurations, with at most nine products (eight for a dress-based outfit). Outfits with two products, or with ten or more, never satisfy the rules.

In [17]:
configurations_base = config["data"]["configurations_base"]
optional_products = config["data"]["optional_products"]
optional_configurations = create_combinations(optional_products)
all_configurations = create_configurations(configurations_base, optional_products)

print("Base configurations:", len(configurations_base))
print("Nonempty subsets of optional products:", len(optional_configurations))
print("Configurations:", len(all_configurations))

Base configurations: 2
Nonempty subsets of optional products: 31
Configurations: 64


In [18]:
df_outfit_products_sel, configurations_count = select_valid_outfits(
    df_outfit_products, "des_product_class", "cod_outfit", all_configurations
)
print("Selected outfits:", df_outfit_products_sel["cod_outfit"].nunique())
print("Configurations that occur:", len(configurations_count))
df_outfit_products_sel.head()

Selected outfits: 3164
Configurations that occur: 45


,cod_outfit,cod_modelo_color,des_product_class,des_filename
2,4019,37000577-10,Footwear,datathon/images/2022_37000577_10.jpg
4,5018,37000577-30,Footwear,datathon/images/2022_37000577_30.jpg
5,2821,37000577-30,Footwear,datathon/images/2022_37000577_30.jpg
7,2811,37000577-30,Footwear,datathon/images/2022_37000577_30.jpg
12,5685,37000577-30,Footwear,datathon/images/2022_37000577_30.jpg


Only a minority of the outfits are selected. The share of each configuration is used later to draw random candidate outfits made of unseen products:

In [19]:
configurations_count_df = pd.DataFrame(
    [(index, " + ".join(all_configurations[index]), count) for index, count in configurations_count.items()],
    columns=["conf_id", "configuration", "outfits"],
).sort_values("outfits", ascending=False)
configurations_count_df["share"] = configurations_count_df["outfits"] / configurations_count_df["outfits"].sum()
configurations_count_df

,conf_id,configuration,outfits,share
1,34,Tops + Bottoms + Footwear + Earrings + Bags,1186,0.374842
7,38,Tops + Bottoms + Footwear + Earrings + Outerwe...,244,0.077118
3,11,"Dresses, jumpsuits and Complete set + Footwear...",229,0.072377
10,2,"Dresses, jumpsuits and Complete set + Footwear...",210,0.066372
9,33,Tops + Bottoms + Footwear + Earrings + Outerwear,181,0.057206
26,10,"Dresses, jumpsuits and Complete set + Footwear...",156,0.049305
0,43,Tops + Bottoms + Footwear + Earrings + Bags + ...,137,0.043300
4,12,"Dresses, jumpsuits and Complete set + Footwear...",134,0.042351
6,32,Tops + Bottoms + Footwear + Earrings,90,0.028445
15,44,Tops + Bottoms + Footwear + Earrings + Bags + ...,69,0.021808


Outfit sizes and class shares of the selected outfits:

In [20]:
get_grouped_counts(df_outfit_products_sel, "cod_outfit", "outfit")

,outfit_size,grouped_outfit_size
0,3,18
1,4,331
2,5,2084
3,6,619
4,7,109
5,8,3


In [21]:
df_outfit_products_sel["des_product_class"].value_counts(normalize=True)

des_product_class
Footwear                               0.194122
Earrings                               0.194122
Bags                                   0.158415
Bottoms                                0.140990
Tops                                   0.140990
Dresses, jumpsuits and Complete set    0.053132
Ring                                   0.039205
Outerwear                              0.038285
Necklace                               0.023192
Glasses                                0.017547
Name: proportion, dtype: float64

## Catalogues

Each image file name starts with the year of its catalogue. Products per image year:

In [22]:
df_products["image_year"] = df_products["des_filename"].map(image_year)
df_products["image_year"].value_counts().sort_index()

image_year
2019    4258
2020      96
2021       3
2022      11
2023    4827
2024      27
Name: count, dtype: int64

Years present in each selected outfit:

In [23]:
years = df_outfit_products_sel["des_filename"].map(image_year)
years.groupby(df_outfit_products_sel["cod_outfit"]).agg(
    lambda values: " + ".join(map(str, sorted(set(values))))
).value_counts().rename_axis("years")

years
2019           1528
2023           1393
2019 + 2020     125
2023 + 2024      93
2022 + 2023      24
2021 + 2023       1
Name: count, dtype: int64

No outfit combines products from 2019 or 2020 with products from 2021 onwards. Years that share an outfit form one catalogue, and candidate outfits are drawn within one catalogue:

In [24]:
selected_wide = df_outfit_products_sel.pivot(
    index="cod_outfit", columns="des_product_class", values="cod_modelo_color"
).reindex(columns=list(PRODUCT_CLASSES))
catalogue = catalogues(df_products, selected_wide)
catalogue.value_counts()

2021-2024    4868
2019-2020    4354
Name: count, dtype: int64

## Products shared between outfits

A test outfit that shares a product with a training outfit leaks information about it. Many products appear in several outfits:

In [25]:
outfits_per_product = df_outfit_products_sel["cod_modelo_color"].value_counts()
print("Products in the selected outfits:", len(outfits_per_product))
print("Products in more than one outfit:", (outfits_per_product > 1).sum())
print("Most outfits for one product:", outfits_per_product.max())

Products in the selected outfits: 4909
Products in more than one outfit: 1849
Most outfits for one product: 373


The cell below assigns outfits to a training and a test set, stratified by configuration, keeps the test outfits that share no product with the training set, and averages over ten random splits.

In [26]:
outfit_products = df_outfit_products_sel.groupby("cod_outfit")["cod_modelo_color"].agg(frozenset)
outfit_configuration = df_outfit_products_sel.groupby("cod_outfit")["des_product_class"].agg(
    lambda classes: tuple(sorted(classes))
)


def leakage_free_test_outfits(train_share, seed):
    """(test outfits, test outfits sharing no product with the training outfits)."""
    rng = np.random.default_rng(seed)
    train = []
    for _, group in outfit_configuration.groupby(outfit_configuration):
        ids = rng.permutation(group.index.to_numpy())
        count = rng.binomial(1, train_share) if len(ids) == 1 else round(train_share * len(ids))
        train.extend(ids[:count])
    seen = frozenset().union(*outfit_products[train])
    test = outfit_products.drop(train)
    return len(test), int(test.map(lambda products: not products & seen).sum())


rows = []
for share in (0.8, 0.5, 0.3, 0.1):
    runs = np.array([leakage_free_test_outfits(share, seed) for seed in range(10)])
    rows.append((share, *runs.mean(axis=0)))
pd.DataFrame(rows, columns=["training share", "test outfits", "without shared products"]).round(1)

,training share,test outfits,without shared products
0,0.8,633.2,0.3
1,0.5,1586.8,2.3
2,0.3,2214.7,8.2
3,0.1,2849.4,80.4


Even with a small training share, only a few test outfits share no product with the training set. The model is therefore trained on all selected outfits and tested on candidate outfits built from products that appear in no training outfit.